# ⚽ Pipeline completo — Football Tracking → Eventos

**Cómo correrlo:** Activá GPU (Entorno de ejecución → GPU) y hacé **Ejecutar todo**.
La celda 1 instala todo y te pide **reiniciar una vez**: reiniciás y volvés a **Ejecutar todo**.
De ahí en más va solo (solo seleccionás el video cuando lo pida).

> Los *warnings* de pip sobre `pointpats/esda/spopt/...` son normales — esos paquetes no se usan.


## 0. GPU


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU — activá una en Entorno de ejecución')


## 1. Instalar (una vez) + reiniciar

Instala dependencias, actualiza ultralytics (para el modelo entrenado), deja Pillow limpio y fija
`numpy<2.1` (lo necesita el clasificador de equipos). Al terminar te pide reiniciar.


In [ ]:
import os
REPO_DIR = '/content/ncf_event_tracker'
FLAG = '/content/.setup_done'
BRANCH = 'events-model'   # rama con --frame-stride y el sidecar .meta.json

if not os.path.exists(FLAG):
    if not os.path.exists(REPO_DIR):
        !git clone -q --branch {BRANCH} https://github.com/pipachiesa/ncf_event_tracker.git {REPO_DIR}
    !pip install -q -r {REPO_DIR}/requirements.txt
    !pip install -q filterpy scipy
    !pip install -q -U ultralytics
    !pip install -q --force-reinstall --no-cache-dir pillow
    !pip install -q 'numpy<2.1'   # para el clasificador de equipos (numba)
    open(FLAG, 'w').close()
    print('\n' + '='*66)
    print('✅ INSTALADO. Ahora: Entorno de ejecución → REINICIAR entorno,')
    print('   y volvé a Ejecutar todo (esta celda se saltea sola).')
    print('='*66)
    raise SystemExit('Reiniciá el entorno y volvé a Ejecutar todo.')

%cd {REPO_DIR}
!git checkout -q {BRANCH} && git pull -q origin {BRANCH}   # asegura la rama correcta
import numpy, ultralytics
print('rama:', open(REPO_DIR+'/.git/HEAD').read().strip().split('/')[-1], '| numpy:', numpy.__version__, '| ultralytics:', ultralytics.__version__)
try:
    import numba; from sports.common.team import TeamClassifier
    print('✅ Entorno OK — clasificador de equipos disponible.')
except Exception as e:
    print(f'⚠️  Clasificador de equipos NO disponible ({type(e).__name__}). El pipeline correrá SIN equipos.')
assert os.path.exists('data_cleanup/main.py')


## 2. Montar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR    = '/content/drive/MyDrive/football_analytics'
TRACKING_DIR = os.path.join(DRIVE_DIR, 'tracking_output')
EVENTS_DIR   = os.path.join(DRIVE_DIR, 'event_output')
for d in (DRIVE_DIR, TRACKING_DIR, EVENTS_DIR): os.makedirs(d, exist_ok=True)
print('Resultados en:', DRIVE_DIR)


## 3. Detector

Busca tu modelo entrenado en Drive y verifica que carga. Si no hay, usa el community `football`.


In [ ]:
import glob
from ultralytics import YOLO
hits = glob.glob(os.path.join(DRIVE_DIR, 'models', '**', 'best.pt'), recursive=True)
if hits:
    DETECTOR = hits[0]
    print('Modelo entrenado:', DETECTOR)
    print('  carga OK, clases:', YOLO(DETECTOR).names)
else:
    DETECTOR = 'football'
    print('⚠️ No hay modelo entrenado en Drive — usando community \'football\'')


## 4. Elegí el video (desde Google Drive)

Lo más simple: subí el `.mp4` a **MyDrive/football_analytics/videos/** y poné
solo el **nombre del archivo** en `VIDEO_FILE`. También podés pegar una ruta
completa de Drive. (Si dejás `VIDEO_FILE=''` cae al viejo modo "subir del navegador".)

In [ ]:
# Subí tu video a MyDrive/football_analytics/videos/ y poné su nombre acá.
# (o pegá una ruta completa de Drive; dejalo '' para subir del navegador)
VIDEO_FILE = 'mi_partido.mp4'

VIDEOS_DIR = os.path.join(DRIVE_DIR, 'videos')
os.makedirs(VIDEOS_DIR, exist_ok=True)

if not VIDEO_FILE:
    from google.colab import files
    print('Seleccioná un .mp4 desde tu compu...')
    up = files.upload()
    VIDEO_PATH = os.path.abspath(list(up.keys())[0])
else:
    # ruta completa tal cual, o solo-nombre -> buscar en la carpeta de videos
    VIDEO_PATH = VIDEO_FILE if os.path.sep in VIDEO_FILE else os.path.join(VIDEOS_DIR, VIDEO_FILE)
    if not os.path.exists(VIDEO_PATH):
        hay = sorted(glob.glob(os.path.join(VIDEOS_DIR, '*.mp4')))
        listado = '\n'.join('  - ' + os.path.basename(v) for v in hay) or '  (carpeta vacía)'
        raise FileNotFoundError(
            f'No encuentro el video: {VIDEO_PATH}\n'
            f'Subí el .mp4 a {VIDEOS_DIR} y poné su nombre en VIDEO_FILE.\n'
            f'Videos que hay ahí ahora:\n{listado}')

VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
print('Video:', VIDEO_PATH)

## 5. Tracking

Borra cualquier CSV viejo primero, así si el tracking falla lo ves (no arrastra datos viejos).

> **Dos modelos distintos a propósito:** el detector entrenado para **jugadores**, y el
> community `football` para el **balón** (el entrenado detecta el balón peor: 54% vs 78%).


In [ ]:
TRACKING_CSV = os.path.join(TRACKING_DIR, VIDEO_NAME + '.csv')
if os.path.exists(TRACKING_CSV): os.remove(TRACKING_CSV)

cmd = (
    f'python data_cleanup/main.py '
    f'--video "{VIDEO_PATH}" '
    f'--output "{TRACKING_DIR}" '
    f'--player-model "{DETECTOR}" '
    f'--ball-model football '   # el modelo entrenado detecta PEOR el balon (54% vs 78%)
    f'--imgsz 1280 '
    f'--pitch-imgsz 1280 '
    f'--ball-conf 0.1 '
    f'--ball-interp-gap 15 '
    f'--track-buffer 150 '
    f'--min-track-frames 12 '
    f'--pitch-model football-field '
    f'--homography-every 5 '
    f'--frame-stride 2'   # 1 de cada 2 frames: compensa la 2da inferencia del balon
)
print(cmd, '\n')
!{cmd}
assert os.path.exists(TRACKING_CSV), '❌ El tracking FALLÓ (no se generó el CSV). Mirá el error de arriba.'
print('✅ Tracking CSV:', TRACKING_CSV)


## 5b. CHEQUEO CLAVE — ¿la pelota se teletransporta?

Test de aceptación de la corrida. Antes, el 24,5% de los movimientos de la
pelota eran **físicamente imposibles** (p90 = 417 m/s = 1502 km/h): el detector
elegía por confianza y saltaba al punto de penal. Con la selección por
continuidad (`_pick_ball`) esto tiene que desplomarse.

> **< 5% = arreglado.** Si sigue arriba del 15%, el fix no entró (¿hiciste
> `git pull` de la rama con el cambio?) y no tiene sentido etiquetar todavía.

In [ ]:
import csv, json

FPS = json.load(open(TRACKING_CSV.rsplit('.',1)[0] + '.meta.json'))['effective_fps']
L_M, W_M = 105.0, 68.0

pts = []
for r in csv.DictReader(open(TRACKING_CSV)):
    if r['Object'] != 'ball':
        continue
    x, y = float(r['X_Pitch']), float(r['Y_Pitch'])
    if x == 0 and y == 0:
        continue
    pts.append((int(r['Frame']), x/12000*L_M, y/7000*W_M))
pts.sort()

sp = []
for i in range(1, len(pts)):
    (f0,x0,y0), (f1,x1,y1) = pts[i-1], pts[i]
    if not 1 <= f1-f0 <= 3:
        continue
    sp.append((((x1-x0)**2 + (y1-y0)**2)**0.5) / ((f1-f0)/FPS))
sp.sort()

pct = lambda q: sp[int(q*(len(sp)-1))]
bad = 100*sum(1 for s in sp if s > 35)/len(sp)   # 35 m/s = tiro potente
print(f'detecciones de pelota: {len(pts)}')
print(f'velocidad p50 {pct(.5):5.1f} m/s | p90 {pct(.9):7.1f} | p99 {pct(.99):7.1f}')
print(f'\nMOVIMIENTOS IMPOSIBLES (>35 m/s): {bad:.1f}%   (antes del fix: 24.5%)')
print('✅ trayectoria sana' if bad < 5 else
      ('🟡 mejoró pero no del todo' if bad < 15 else
       '❌ el fix NO entró — revisá que el git pull traiga _pick_ball'))
cand = TRACKING_CSV.rsplit('.', 1)[0] + '_ball_candidates.csv'
if os.path.exists(cand):
    n = sum(1 for _ in open(cand)) - 1
    print(f'\ncandidatos de pelota guardados: {n} (para ball_viterbi.py) ✅')
else:
    print('\n❌ no se generaron candidatos — el git pull no trajo la version nueva')

## 6. Generar eventos


In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'data_cleanup'))
from lib.match import Match
match = Match()
match.import_raw_data(os.path.dirname(TRACKING_CSV) + os.sep, os.path.basename(TRACKING_CSV))
print(f'Importados {match.frames} frames y {len(match.players)} objetos.')
events = match.generate_events()
print('Resumen:', events.summary())
EVENTS_CSV = os.path.join(EVENTS_DIR, VIDEO_NAME + '_events.csv')
events.export(path=EVENTS_DIR + os.sep, file_name=os.path.basename(EVENTS_CSV))
print('✅ Eventos:', EVENTS_CSV)


## 7. Chequeo — fragmentación de jugadores


In [ ]:
import csv
from collections import Counter
life = Counter()
for r in csv.DictReader(open(TRACKING_CSV)):
    if r['Object'] == 'player': life[r['Object ID']] += 1
print('IDs de jugador distintos:', len(life), ' (menos = mejor; con el modelo chico eran ~184)')


## 8. Visualización


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Circle
df = pd.read_csv(EVENTS_CSV)
print(df['Type'].value_counts())

def draw_pitch(ax):
    ax.add_patch(plt.Rectangle((0,0),1,1,fill=False,color='black',lw=2))
    ax.plot([0.5,0.5],[0,1],color='black',lw=1)
    ax.add_patch(Circle((0.5,0.5),0.083,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.add_patch(plt.Rectangle((0.84,0.21),0.16,0.58,fill=False,color='black',lw=1))
    ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
    ax.set_aspect(68.0/105.0); ax.axis('off')

pdf = df.dropna(subset=['Start X','Start Y'])
types = sorted(pdf['Type'].unique())
pal = list(plt.cm.tab10.colors)
colors = {t: pal[i % len(pal)] for i,t in enumerate(types)}
fig, ax = plt.subplots(figsize=(12,8))
ax.add_patch(plt.Rectangle((0,0),1,1,color='#3a8a3a',alpha=0.12,zorder=0)); draw_pitch(ax)
for t in types:
    s = pdf[pdf['Type']==t]
    ax.scatter(s['Start X'], s['Start Y'], s=120, color=colors[t], edgecolors='black', linewidths=0.6, label=f'{t} ({len(s)})', zorder=3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.02), ncol=4, frameon=False)
ax.set_title(f'Eventos detectados — {VIDEO_NAME}')
fig_path = os.path.join(EVENTS_DIR, VIDEO_NAME + '_event_map.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight'); plt.show()
print('Mapa:', fig_path)


## 9. Descargar los CSV


In [ ]:
from google.colab import files

# Los TRES archivos son necesarios aguas abajo:
#   tracking.csv            -> el tracking
#   tracking.meta.json      -> fps efectivo (sin esto los tiempos salen mal)
#   *_ball_candidates.csv   -> candidatos de pelota para ball_viterbi.py
for path in (TRACKING_CSV,
             TRACKING_CSV.rsplit('.', 1)[0] + '.meta.json',
             TRACKING_CSV.rsplit('.', 1)[0] + '_ball_candidates.csv',
             EVENTS_CSV):
    if os.path.exists(path):
        files.download(path)
    else:
        print('FALTA:', path)

print('''
Guardalos asi (mismo nombre base, en la carpeta del partido):
  ~/football_data/matches/<partido>/tracking.csv
  ~/football_data/matches/<partido>/tracking.meta.json
  ~/football_data/matches/<partido>/tracking_ball_candidates.csv
''')